# 실습 2-2. 불량 분석과 대시보드

실습 2-1 에서 만든 정제본으로 **어떤 성형 조건이 불량을 만드는지** 찾고, 불량 분류 모델을 만든 뒤,
일자별 불량률 추이를 대시보드 패널로 그림.

- **기본 트랙**: 셀을 차례로 실행하고 마지막의 "해석 세 문장"을 워크북에 적음.
- **심화 트랙**: `# 심화` 표시가 있는 부분을 바꿔 불량 분류 F1 점수 0.75 이상을 확인하고 더 올려 봄.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
while not (ROOT / "common").exists():      # 저장소 최상위 폴더 찾기
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from common.plot_style import use_korean_font
use_korean_font()

DATA = ROOT / "data" / "session2"
OUT = DATA / "work"                        # 내 산출물이 저장되는 곳
OUT.mkdir(exist_ok=True)
pd.set_option("display.width", 160)

In [ ]:
src = OUT / "injection_clean.csv"
if not src.exists():                      # 실습 2-1 을 끝내지 못했으면 제공 정제본을 사용
    src = DATA / "injection_clean_provided.csv"
df = pd.read_csv(src, parse_dates=["timestamp"])
df["date"] = df["timestamp"].dt.strftime("%m-%d")
FEATURES = ["mold_temp_c", "melt_temp_c", "inj_pressure_mpa", "hold_pressure_mpa",
            "hold_time_s", "cooling_time_s", "cycle_time_s", "ambient_temp_c"]
KO = {"mold_temp_c": "금형온도", "melt_temp_c": "수지온도", "inj_pressure_mpa": "사출압",
      "hold_pressure_mpa": "보압", "hold_time_s": "보압 시간", "cooling_time_s": "냉각 시간",
      "cycle_time_s": "사이클타임", "ambient_temp_c": "주변 온도", "cooling_ratio": "냉각 비율"}
DEFECT_KO = {"none": "양품", "flow_mark": "흐름자국", "sink": "수축", "short_shot": "미성형"}
print(src.name, df.shape, f" 불량률 {df['is_defect'].mean()*100:.1f}%")
df["defect_type"].map(DEFECT_KO).value_counts()

## 1. 언제, 어느 설비에서 불량이 많았나

In [ ]:
pivot = df.pivot_table(index="date", columns="machine_id", values="is_defect", aggfunc="mean") * 100
ax = pivot.plot(marker="o", figsize=(10, 3.4))
ax.set_ylabel("불량률 (%)"); ax.set_title("일자·설비별 불량률"); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()
pivot.round(1)

> 불량이 몰린 날이 설비마다 다름. 그날 조건이 평소와 어떻게 달랐는지 봄.

In [ ]:
daily = df.groupby(["machine_id", "date"])[["mold_temp_c", "hold_pressure_mpa", "cooling_time_s", "inj_pressure_mpa"]].mean()
fig, axes = plt.subplots(1, 4, figsize=(14, 3.2))
for ax, col in zip(axes, daily.columns):
    daily[col].unstack(0).plot(ax=ax, marker=".", legend=(col == "mold_temp_c"))
    ax.set_title("일평균 " + KO[col]); ax.set_xlabel(""); ax.tick_params(axis="x", rotation=60)
plt.tight_layout(); plt.show()

## 2. 조건과 불량의 관계

양품과 불량 유형별로 조건 분포를 겹쳐 봄.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.4))
for ax, (col, kind) in zip(axes, [("mold_temp_c", "flow_mark"), ("hold_pressure_mpa", "sink"), ("cooling_time_s", "sink")]):
    bins = np.linspace(df[col].min(), df[col].max(), 45)
    ax.hist(df.loc[df.defect_type == "none", col], bins=bins, alpha=0.6, label="양품", color="#2471a3")
    ax.hist(df.loc[df.defect_type == kind, col], bins=bins, alpha=0.8, label=DEFECT_KO[kind], color="#c0392b")
    ax.set_title(f"{KO[col]}  vs  {DEFECT_KO[kind]}"); ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
# 조건을 구간으로 나눠 구간별 불량률을 봄
fig, axes = plt.subplots(1, 3, figsize=(13, 3.2))
for ax, col in zip(axes, ["mold_temp_c", "hold_pressure_mpa", "cooling_time_s"]):
    g = df.groupby(pd.cut(df[col], 12), observed=True)["is_defect"].agg(["mean", "size"])
    g = g[g["size"] >= 15]
    ax.bar([f"{iv.mid:.1f}" for iv in g.index], g["mean"] * 100, color="#af601a")
    ax.set_title(f"{KO[col]} 구간별 불량률 (%)"); ax.tick_params(axis="x", rotation=60)
plt.tight_layout(); plt.show()

> 금형온도가 낮을수록, 보압이 낮을수록, 냉각 시간이 짧을수록 불량률이 급히 오름.
> 가상 공장 화면에서 슬라이더로 조건을 바꿨을 때 본 현상과 같은지 비교함.

## 3. 불량 분류 모델

성형 조건만 보고 "이 샷은 불량일 것"을 맞히는 모델을 만듦. 전체의 75% 로 학습하고 나머지 25% 로 평가함.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, f1_score, precision_score, recall_score

USE = list(FEATURES)                      # 심화: 특성을 빼거나 "cooling_ratio" 를 더해 봄
X_train, X_test, y_train, y_test = train_test_split(df[USE], df["is_defect"], test_size=0.25,
                                                    random_state=0, stratify=df["is_defect"])
model = RandomForestClassifier(n_estimators=200,      # 심화: 나무 수
                               min_samples_leaf=3,    # 심화: 잎의 최소 샘플 수
                               random_state=0, n_jobs=-1)
model.fit(X_train, y_train)
prob = model.predict_proba(X_test)[:, 1]

In [ ]:
THRESHOLD = 0.5                           # 심화: 불량으로 판정하는 기준 확률
pred = (prob >= THRESHOLD).astype(int)
cm = confusion_matrix(y_test, pred)
result = pd.DataFrame(cm, index=["실제 양품", "실제 불량"], columns=["양품으로 예측", "불량으로 예측"])
print(result, "\n")
print(f"정밀도 {precision_score(y_test, pred):.3f}   (불량이라 한 것 중 진짜 불량)")
print(f"재현율 {recall_score(y_test, pred):.3f}   (진짜 불량 중 잡아낸 것)")
print(f"F1 점수 {f1_score(y_test, pred):.3f}   (목표 0.75 이상)")

In [ ]:
# 기준 확률을 바꾸면 과검(양품을 불량으로)과 미검(불량을 양품으로)이 어떻게 달라지나
rows = []
for th in np.arange(0.1, 0.95, 0.1):
    p = (prob >= th).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, p).ravel()
    rows.append({"기준 확률": round(th, 1), "과검(건)": fp, "미검(건)": fn, "F1": round(f1_score(y_test, p), 3)})
pd.DataFrame(rows).set_index("기준 확률")

## 4. 변수 중요도: 모델은 무엇을 보고 판단했나

In [ ]:
imp = pd.Series(model.feature_importances_, index=[KO[c] for c in USE]).sort_values()
ax = imp.plot.barh(figsize=(7, 3.4), color="#1e8449")
ax.set_title("변수 중요도"); plt.tight_layout(); plt.show()
imp.sort_values(ascending=False).round(3)

> 상위 항목이 금형온도·보압·냉각 시간(사이클타임)인지 확인함.
> 중요도는 "이 데이터에서 판정에 많이 쓰였다"는 뜻이지 원인을 증명하는 값이 아님. 그래서 2절의 그림, 공정 지식과 함께 봄.

## 5. 대시보드 패널

현장 모니터에 띄운다고 생각하고 네 칸짜리 패널로 정리함.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 7))
fig.suptitle("사출 라인 불량 현황", fontsize=16)

ax = axes[0, 0]
total = df.groupby("date")["is_defect"].mean() * 100
ax.plot(total.index, total.values, marker="o", color="#c0392b", lw=2)
ax.axhline(5, color="gray", ls="--", lw=1); ax.text(0, 5.6, "관리 기준 5%", color="gray")
ax.set_title("일자별 불량률 (%)"); ax.tick_params(axis="x", rotation=45); ax.grid(alpha=0.3)

ax = axes[0, 1]
kinds = df[df.is_defect == 1].pivot_table(index="date", columns="defect_type", values="is_defect", aggfunc="size", fill_value=0)
kinds = kinds.reindex(total.index, fill_value=0).rename(columns=DEFECT_KO)
kinds.columns.name = None
kinds.plot.bar(stacked=True, ax=ax, color=["#d68910", "#6c3483", "#2e86c1"][:kinds.shape[1]])
ax.set_title("일자별 불량 유형 (건)"); ax.set_xlabel(""); ax.tick_params(axis="x", rotation=45)

ax = axes[1, 0]
pivot = df.pivot_table(index="date", columns="machine_id", values="is_defect", aggfunc="mean") * 100
im = ax.imshow(pivot.T.values, aspect="auto", cmap="Reds", vmin=0, vmax=60)
ax.set_xticks(range(len(pivot.index))); ax.set_xticklabels(pivot.index, rotation=45)
ax.set_yticks(range(3)); ax.set_yticklabels(pivot.columns)
for i in range(pivot.shape[1]):
    for j in range(pivot.shape[0]):
        v = pivot.values[j, i]
        ax.text(j, i, f"{v:.0f}", ha="center", va="center", fontsize=9, color="white" if v > 35 else "black")
ax.set_title("설비·일자별 불량률 (%)")

ax = axes[1, 1]
imp.tail(5).plot.barh(ax=ax, color="#1e8449"); ax.set_title("불량에 영향이 큰 조건 (상위 5)")

plt.tight_layout(); fig.savefig(OUT / "dashboard.png", dpi=130); plt.show()

## 6. 도장 부스 데이터 (2교시 사례 확인용)

부스 습도·온도·도료 점도·도막 두께와 흐름·핀홀 불량의 관계를 같은 방법으로 봄.

In [ ]:
paint = pd.read_csv(DATA / "paint_booth.csv", parse_dates=["timestamp"])
PK = {"booth_humidity_pct": "부스 습도(%)", "booth_temp_c": "부스 온도(℃)", "viscosity_s": "도료 점도(s)", "film_thickness_um": "도막 두께(㎛)"}
fig, axes = plt.subplots(1, 4, figsize=(15, 3.2))
for ax, col in zip(axes, PK):
    for kind, name, color in (("pinhole", "핀홀", "#6c3483"), ("sag", "흐름", "#d68910")):
        g = paint.assign(hit=(paint.defect_type == kind)).groupby(pd.cut(paint[col], 10), observed=True)["hit"].agg(["mean", "size"])
        g = g[g["size"] >= 15]
        ax.plot([iv.mid for iv in g.index], g["mean"] * 100, marker="o", label=name, color=color)
    ax.set_title(PK[col] + " 구간별 불량률 (%)"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 7. 산출물 저장과 해석 세 문장

In [ ]:
summary = pd.DataFrame({
    "항목": ["평가 데이터 수", "기준 확률", "정밀도", "재현율", "F1 점수", "중요도 1위", "중요도 2위", "중요도 3위"],
    "값": [len(y_test), THRESHOLD, round(precision_score(y_test, pred), 3), round(recall_score(y_test, pred), 3),
           round(f1_score(y_test, pred), 3), *imp.sort_values(ascending=False).index[:3]],
})
summary.to_csv(OUT / "model_result.csv", index=False, encoding="utf-8-sig")
summary

### 워크북에 적기: 해석 세 문장

1. **언제·어디서**: 불량률이 가장 높았던 날과 설비는 ( ) 이고, 그날 ( ) 조건이 평소와 달랐음.
2. **무엇이**: 불량에 영향이 큰 조건은 ( ), ( ), ( ) 순이었음.
3. **그래서**: 우리 라인이라면 ( ) 값을 ( ) 범위로 관리하고, ( ) 를 대시보드에 띄우겠음.

저장된 파일: `data/session2/work/model_result.csv`, `data/session2/work/dashboard.png`